# Utils

This notebook is only used to try out different things without changing the other notebooks too much.

In [107]:
import os
import shutil
import h5py
import numpy as np
import sys

sys.path.append("..")
sys.path.append("../code")

from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.cadlib.visualize import vec2CADsolid
from OCC.Core.BRepCheck import BRepCheck_Analyzer
from OCC.Extend.DataExchange import write_step_file
from OCC.Core.STEPControl import STEPControl_Reader
from OCC.Core.StlAPI import StlAPI_Writer
from OCC.Core.BRepMesh import BRepMesh_IncrementalMesh

In [108]:
def copy_to_temp(dataset, idx):
    cad_seq_path = dataset.get_cad_seq_path(idx)
    
    temp_dir = "../data/temporary"
    if os.path.exists(temp_dir):
        shutil.rmtree(temp_dir)
    os.makedirs(temp_dir)

    dest_path = os.path.join(temp_dir, os.path.basename(cad_seq_path))

    shutil.copy(cad_seq_path, dest_path)
    print(f"Copied {cad_seq_path} to {dest_path}")
    return dest_path

def change_keys(h5_file):
    """Changes the keys from 'vec' to 'out_vec' in order to be able to show the sample using show.py"""
    with h5py.File(h5_file, 'r+') as hf:
        if 'vec' in hf:
            data = hf['vec'][:]
            hf.create_dataset('out_vec', data=data)
            del hf['vec']
            print(f"Changed keys from 'vec' to 'out_vec' in {h5_file}")

def export2step(h5_path):
    filter = True
    save_path = os.path.join(*h5_path.split("/")[:-1], os.path.splitext(os.path.basename(h5_path))[0] + '.step')

    with h5py.File(h5_path, 'r') as fp:
        seq = fp['out_vec'][:].astype(np.float32)

        out_shape = vec2CADsolid(seq)

        if filter:
            analyzer = BRepCheck_Analyzer(out_shape)
            if not analyzer.IsValid():
                print(f"CAD-sequence of {os.path.basename(pc_path)} is invalid.")

        write_step_file(out_shape, save_path)
    return save_path

def step2stl(step_path):

    save_path = os.path.join(*step_path.split("/")[:-1], os.path.splitext(os.path.basename(step_path))[0] + '.stl')
    step_reader = STEPControl_Reader()
    step_reader.ReadFile(step_path)
    step_reader.TransferRoots()
    shape = step_reader.OneShape()

    BRepMesh_IncrementalMesh(shape, 0.1)

    stl_writer = StlAPI_Writer()
    stl_writer.Write(shape, save_path)
    print(f"Wrote stl file to {save_path}")

def visualize_gt(dataset, idx):
    dest_path_h5 = copy_to_temp(dataset, index)
    change_keys(dest_path_h5)
    step_path = export2step(dest_path_h5)
    step2stl(step_path)

In [109]:
index = 3

In [110]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')

In [111]:
visualize_gt(dataset, index)

Copied ../data/cad_vec/0023/00239323.h5 to ../data/temporary/00239323.h5
Changed keys from 'vec' to 'out_vec' in ../data/temporary/00239323.h5

*******************************************************************
******        Statistics on Transfer (Write)                 ******

*******************************************************************
******        Transfer Mode = 0  I.E.  As Is       ******
******        Transferring Shape, ShapeType = 2                      ******
** WorkSession : Sending all data
 Step File Name : ../data/temporary/00239323.step(552 ents)  Write  Done
Wrote stl file to ../data/temporary/00239323.stl


In [1]:
import torch

In [7]:
lol = torch.load("../models/trained_models/github_model_pn/latest.pth", weights_only = True, map_location=torch.device('cpu'))

In [16]:
print(type(lol))
print(lol.keys())
print(lol['model_state_dict'].keys())

<class 'dict'>
dict_keys(['clock', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict'])
odict_keys(['SA_modules.0.mlps.0.0.weight', 'SA_modules.0.mlps.0.1.weight', 'SA_modules.0.mlps.0.1.bias', 'SA_modules.0.mlps.0.1.running_mean', 'SA_modules.0.mlps.0.1.running_var', 'SA_modules.0.mlps.0.1.num_batches_tracked', 'SA_modules.0.mlps.0.3.weight', 'SA_modules.0.mlps.0.4.weight', 'SA_modules.0.mlps.0.4.bias', 'SA_modules.0.mlps.0.4.running_mean', 'SA_modules.0.mlps.0.4.running_var', 'SA_modules.0.mlps.0.4.num_batches_tracked', 'SA_modules.0.mlps.0.6.weight', 'SA_modules.0.mlps.0.7.weight', 'SA_modules.0.mlps.0.7.bias', 'SA_modules.0.mlps.0.7.running_mean', 'SA_modules.0.mlps.0.7.running_var', 'SA_modules.0.mlps.0.7.num_batches_tracked', 'SA_modules.1.mlps.0.0.weight', 'SA_modules.1.mlps.0.1.weight', 'SA_modules.1.mlps.0.1.bias', 'SA_modules.1.mlps.0.1.running_mean', 'SA_modules.1.mlps.0.1.running_var', 'SA_modules.1.mlps.0.1.num_batches_tracked', 'SA_modules.1.mlps.0.3.weigh

In [ ]:
dict_keys(['clock', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict'])
odict_keys(['SA_modules.0.mlps.0.0.weight', 'SA_modules.0.mlps.0.1.weight', 'SA_modules.0.mlps.0.1.bias', 'SA_modules.0.mlps.0.1.running_mean', 'SA_modules.0.mlps.0.1.running_var', 'SA_modules.0.mlps.0.1.num_batches_tracked', 'SA_modules.0.mlps.0.3.weight', 'SA_modules.0.mlps.0.4.weight', 'SA_modules.0.mlps.0.4.bias', 'SA_modules.0.mlps.0.4.running_mean', 'SA_modules.0.mlps.0.4.running_var', 'SA_modules.0.mlps.0.4.num_batches_tracked', 'SA_modules.0.mlps.0.6.weight', 'SA_modules.0.mlps.0.7.weight', 'SA_modules.0.mlps.0.7.bias', 'SA_modules.0.mlps.0.7.running_mean', 'SA_modules.0.mlps.0.7.running_var', 'SA_modules.0.mlps.0.7.num_batches_tracked', 'SA_modules.1.mlps.0.0.weight', 'SA_modules.1.mlps.0.1.weight', 'SA_modules.1.mlps.0.1.bias', 'SA_modules.1.mlps.0.1.running_mean', 'SA_modules.1.mlps.0.1.running_var', 'SA_modules.1.mlps.0.1.num_batches_tracked', 'SA_modules.1.mlps.0.3.weight', 'SA_modules.1.mlps.0.4.weight', 'SA_modules.1.mlps.0.4.bias', 'SA_modules.1.mlps.0.4.running_mean', 'SA_modules.1.mlps.0.4.running_var', 'SA_modules.1.mlps.0.4.num_batches_tracked', 'SA_modules.1.mlps.0.6.weight', 'SA_modules.1.mlps.0.7.weight', 'SA_modules.1.mlps.0.7.bias', 'SA_modules.1.mlps.0.7.running_mean', 'SA_modules.1.mlps.0.7.running_var', 'SA_modules.1.mlps.0.7.num_batches_tracked', 'SA_modules.2.mlps.0.0.weight', 'SA_modules.2.mlps.0.1.weight', 'SA_modules.2.mlps.0.1.bias', 'SA_modules.2.mlps.0.1.running_mean', 'SA_modules.2.mlps.0.1.running_var', 'SA_modules.2.mlps.0.1.num_batches_tracked', 'SA_modules.2.mlps.0.3.weight', 'SA_modules.2.mlps.0.4.weight', 'SA_modules.2.mlps.0.4.bias', 'SA_modules.2.mlps.0.4.running_mean', 'SA_modules.2.mlps.0.4.running_var', 'SA_modules.2.mlps.0.4.num_batches_tracked', 'SA_modules.2.mlps.0.6.weight', 'SA_modules.2.mlps.0.7.weight', 'SA_modules.2.mlps.0.7.bias', 'SA_modules.2.mlps.0.7.running_mean', 'SA_modules.2.mlps.0.7.running_var', 'SA_modules.2.mlps.0.7.num_batches_tracked', 'SA_modules.3.mlps.0.0.weight', 'SA_modules.3.mlps.0.1.weight', 'SA_modules.3.mlps.0.1.bias', 'SA_modules.3.mlps.0.1.running_mean', 'SA_modules.3.mlps.0.1.running_var', 'SA_modules.3.mlps.0.1.num_batches_tracked', 'SA_modules.3.mlps.0.3.weight', 'SA_modules.3.mlps.0.4.weight', 'SA_modules.3.mlps.0.4.bias', 'SA_modules.3.mlps.0.4.running_mean', 'SA_modules.3.mlps.0.4.running_var', 'SA_modules.3.mlps.0.4.num_batches_tracked', 'SA_modules.3.mlps.0.6.weight', 'SA_modules.3.mlps.0.7.weight', 'SA_modules.3.mlps.0.7.bias', 'SA_modules.3.mlps.0.7.running_mean', 'SA_modules.3.mlps.0.7.running_var', 'SA_modules.3.mlps.0.7.num_batches_tracked', 'fc_layer.0.weight', 'fc_layer.0.bias', 'fc_layer.2.weight', 'fc_layer.2.bias', 'fc_layer.4.weight', 'fc_layer.4.bias'])